### Import Library

In [5]:
import os
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis, linregress
from sklearn.utils import shuffle

### Parameter

In [6]:
senyawa_list = ['butana', 'spirtus']
n_files = 40

### Fungsi Hitung Fitur

In [7]:
def calc_features(signal, time):
    auc = np.trapezoid(signal)
    mean = np.mean(signal)
    max_v = np.max(signal)
    sk = skew(signal)
    ku = kurtosis(signal)

    positive_data = signal[signal > 0]
    positive_time = time[signal > 0]

    if len(positive_data) < 2:
        return np.nan, "Not enough positive signal points to calculate decay rate."

    # Log transform the signal
    log_data = np.log(positive_data)

    # Perform linear regression on log-transformed signal
    slope, intercept, r_value, p_value, std_err = linregress(positive_time, log_data)

    # The decay rate (lambda) is the negative of the slope
    decay_rate = -slope
    return [auc, mean, max_v, sk, ku, decay_rate]

### Ekstraksi Fitur

In [8]:
features_norm = []
features_filt_norm = []

for senyawa in senyawa_list:
    for i in range(1, n_files + 1):

        # =============== NORMALIZED BIASA ===============
        path_norm = f"../Data/normalized_nonfiltered/{senyawa}/{senyawa}_normalized_nonfiltered_{i}.csv"
        df_norm = pd.read_csv(path_norm)
        tvoc_norm = calc_features(df_norm['TVOC (ppb)'], df_norm['Time (s)'])
        tcd_norm  = calc_features(df_norm['TCD (mV)'], df_norm['Time (s)'])
        features_norm.append(tvoc_norm + tcd_norm + [senyawa])

        # =============== FILTERED + NORMALIZED ===============
        path_fnorm = f"../Data/normalized_filtered/{senyawa}/{senyawa}_normalized_filtered_{i}.csv"
        df_fnorm = pd.read_csv(path_fnorm)
        tvoc_fnorm = calc_features(df_fnorm['TVOC (ppb)'], df_fnorm['Time (s)'])
        tcd_fnorm  = calc_features(df_fnorm['TCD (mV)'], df_fnorm['Time (s)'])
        features_filt_norm.append(tvoc_fnorm + tcd_fnorm + [senyawa])

### Simpan CSV

In [10]:
columns1 = [
    "TVOC_AUC", "TVOC_Mean", "TVOC_Max", "TVOC_Skew", "TVOC_Kurtosis", "TVOC_Decay",
    "TCD_AUC", "TCD_Mean", "TCD_Max", "TCD_Skew", "TCD_Kurtosis", "TCD_Decay",
    "Label"
]

# columns = [
#     "TVOC_AUC", "TVOC_Mean", "TVOC_Max", "TVOC_Skew", "TVOC_Kurtosis",
#     "TCD_AUC", "TCD_Mean", "TCD_Max", "TCD_Skew", "TCD_Kurtosis",
#     "Label",
# ]

df_norm = pd.DataFrame(features_norm, columns=columns1)
df_fnorm = pd.DataFrame(features_filt_norm, columns=columns1)

df_norm = shuffle(df_norm, random_state=42)
df_fnorm = shuffle(df_fnorm, random_state=42)

os.makedirs("features_data", exist_ok=True)

df_norm.to_csv("features_data/Combined_normalized_features.csv", index=False)
df_fnorm.to_csv("features_data/Combined_filtered_normalized_features.csv", index=False)

# Pisah TVOC/TCD
for df, prefix in zip([df_norm, df_fnorm], ['Normalized', 'Filtered_Normalized']):
    df[[
        "TVOC_AUC", "TVOC_Mean", "TVOC_Max", "TVOC_Skew", "TVOC_Kurtosis", "TVOC_Decay", "Label"
    ]].to_csv(f"features_data/TVOC_{prefix}_features.csv", index=False)

    # df[[
    #     "TVOC_AUC", "TVOC_Mean", "TVOC_Max", "TVOC_Skew", "TVOC_Kurtosis", "Label"
    # ]].to_csv(f"features_data/TVOC_{prefix}_features.csv", index=False)

    # df[[
    #     "TCD_AUC", "TCD_Mean", "TCD_Max", "TCD_Skew", "TCD_Kurtosis", "Label"
    # ]].to_csv(f"features_data/TCD_{prefix}_features.csv", index=False)

    df[[
        "TCD_AUC", "TCD_Mean", "TCD_Max", "TCD_Skew", "TCD_Kurtosis", "TCD_Decay", "Label"
    ]].to_csv(f"features_data/TCD_{prefix}_features.csv", index=False)

print("✅ Semua file fitur Normalized & Filtered_Normalized selesai diekstrak & disimpan.")

✅ Semua file fitur Normalized & Filtered_Normalized selesai diekstrak & disimpan.
